In [ ]:
!pip install os

In [ ]:
!pip install gradio opencv-python tensorflow

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')


FIRE_PATH = "/content/drive/MyDrive/fire dataset/fire"
NON_FIRE_PATH = "/content/drive/MyDrive/fire dataset/non_fire"

print("fire:", len(os.listdir(FIRE_PATH)))
print("non_fire:", len(os.listdir(NON_FIRE_PATH)))

In [ ]:
!pip install tensorflow opencv-python gradio

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os, shutil


BASE_DATASET = "/content/dataset"
FIRE_TEMP = BASE_DATASET + "/fire"
NON_FIRE_TEMP = BASE_DATASET + "/non_fire"

os.makedirs(FIRE_TEMP, exist_ok=True)
os.makedirs(NON_FIRE_TEMP, exist_ok=True)

# Copy FIRE images
for img in os.listdir(FIRE_PATH):
    src = os.path.join(FIRE_PATH, img)
    if os.path.isfile(src):
        shutil.copy(src, FIRE_TEMP)

# Copy NON-FIRE images
for img in os.listdir(NON_FIRE_PATH):
    src = os.path.join(NON_FIRE_PATH, img)
    if os.path.isfile(src):
        shutil.copy(src, NON_FIRE_TEMP)

print("✅ Dataset prepared")

# ===== CONFIG =====
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 10

# ===== DATA GENERATOR =====
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    BASE_DATASET,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

val_data = datagen.flow_from_directory(
    BASE_DATASET,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

# ===== CNN MODEL =====
model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ===== TRAIN =====
model.fit(train_data, validation_data=val_data, epochs=EPOCHS)

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/fire_detection_model.h5"
model.save(MODEL_PATH)

print("✅ Model saved as fire_detection_model.h5 in Google Drive")

In [ ]:
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2

# ===== LOAD MODEL =====
MODEL_PATH = "/content/drive/MyDrive/fire_detection_model.h5"
model = tf.keras.models.load_model(MODEL_PATH)

IMG_SIZE = 128

# ===== FIRE DETECTION FUNCTION =====
def fire_detection(image):

    if image is None:
        return "", "", ""

    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    image = np.expand_dims(image, axis=0)

    pred = model.predict(image)[0][0]

    # ===== FIRE DETECTED =====
    if pred < 0.5:

        popup = """
        <div id="overlay" style="
            position:fixed;
            top:0; left:0;
            width:100%; height:100%;
            background:rgba(0,0,0,0.75);
            z-index:9999;
            display:flex;
            align-items:center;
            justify-content:center;">

            <div style="
                background:#ff1a1a;
                color:white;
                padding:35px;
                border-radius:15px;
                width:420px;
                text-align:center;
                font-size:22px;
                box-shadow:0px 0px 40px red;">

                🚨 <b>EMERGENCY ALERT</b> 🚨 <br><br>
                🔥 <b>FIRE DETECTED</b> 🔥

                <br><br>

                <button onclick="document.getElementById('overlay').style.display='none'"
                style="
                background:black;
                color:white;
                border:none;
                padding:10px 20px;
                font-size:16px;
                border-radius:8px;
                cursor:pointer;">
                Acknowledge
                </button>

            </div>
        </div>
        """

        status = """
        <div style="
        background:#ff4d4d;
        color:white;
        padding:20px;
        border-radius:12px;
        font-size:22px;
        font-weight:bold;
        text-align:center;">
        🔥 FIRE DETECTED
        </div>
        """

        return "UNSAFE", popup, status

    # ===== NO FIRE =====
    else:

        safe_status = """
        <div style="
        background:#2ecc71;
        color:white;
        padding:20px;
        border-radius:12px;
        font-size:22px;
        font-weight:bold;
        text-align:center;">
        ✅ NO FIRE DETECTED
        </div>
        """

        return "SAFE", "", safe_status


# ===== GRADIO INTERFACE =====
app = gr.Interface(
    fn=fire_detection,

    inputs=gr.Image(
        sources=["upload", "webcam"],
        type="numpy",
        label="Upload Image or Capture from Webcam"
    ),

    outputs=[
        gr.Textbox(label="Result"),
        gr.HTML(label="Emergency Popup"),
        gr.HTML(label="System Status")
    ],

    title="🔥 Fire Detection System",
    description="Upload an image or capture from webcam to detect fire using AI."
)

app.launch(debug=True)